# Apprentissage par transfert avec un modèle fondamental d'OT pour la détection de nappes de pétrole dans un contexte de données limitées

Ce tutoriel démontre comment utiliser un modèle fondamental d'Observation de la Terre (OT) pré-entraîné globalement pour effectuer une classification au niveau de l'image sur des données SAR Sentinel-1.

**Énoncé du problème :**  
Étant donné une image SAR Sentinel-1 en rétrodiffusion Sigma0 (polarisations VV et VH), exprimée en unités de décibels (dB), la tâche consiste à déterminer si l'image :
- contient une nappe de pétrole (*classe = oil*),
- représente de l'eau propre (*classe = clean*) ou contient un phénomène ressemblant à une nappe de pétrole (*classe = lookalike*).
  
**Contrainte :** Nous ne disposons que d'environ 100 images étiquetées de chaque classe pour commencer. Ce domaine de problème est connu sous le nom de contexte de données limitées ou régime limité en données.

**Approche proposée :**  
Nous construisons un classificateur supervisé au-dessus d'un modèle fondamental d'OT pré-entraîné et l'adaptons à cette tâche en entraînant une tête de classification légère utilisant des images SAR étiquetées.

**Choix du modèle :**  
Dans ce tutoriel, nous utilisons le modèle pré-entraîné SATLAS de *Allen Institute*, qui fournit des représentations entraînées globalement pour les données d'observation de la Terre. Voir le dépôt officiel pour les détails sur l'architecture du modèle, les données d'entraînement et les licences :
https://github.com/allenai/satlaspretrain_models et https://huggingface.co/allenai/satlas-pretrain

**Jeu de données :**  
Le jeu de données utilisé dans ce tutoriel est obtenu à partir de Zenodo (DOI: 10.5281/zenodo.8346859). Veuillez vous référer à la source originale (https://zenodo.org/records/8346860) pour les conditions de licence et les restrictions d'utilisation. Notez que le jeu de données est plus grand, mais j'ai choisi un petit sous-ensemble aléatoire du jeu de données d'entraînement pour les besoins de ce tutoriel.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os

In [ ]:
import OilSpillClassification as oilcls

In [ ]:
train_images_dir_oil = r"C:\Users\user\Documents\oilspill\dataset\train\oil"
train_images_dir_look= r"C:\Users\user\Documents\oilspill\dataset\train\lookalike"
train_images_dir_clean = r"C:\Users\user\Documents\oilspill\dataset\train\clean"
train_df = {'images_dir_oil': train_images_dir_oil,
            'images_dir_lookalike': train_images_dir_look,
            'images_dir_clean': train_images_dir_clean}

val_images_dir_oil = r"C:\Users\user\Documents\oilspill\dataset\val\oil"
val_images_dir_look = r"C:\Users\user\Documents\oilspill\dataset\val\lookalike"
val_images_dir_clean = r"C:\Users\user\Documents\oilspill\dataset\val\clean"
val_df = {'images_dir_oil': val_images_dir_oil,
            'images_dir_lookalike': val_images_dir_look,
            'images_dir_clean': val_images_dir_clean}

test_images_dir_oil = r"C:\Users\user\Documents\oilspill\dataset\test\oil"
test_images_dir_look = r"C:\Users\user\Documents\oilspill\dataset\test\lookalike"
test_images_dir_clean = r"C:\Users\user\Documents\oilspill\dataset\test\clean"
test_df = {'images_dir_oil': test_images_dir_oil,
          'images_dir_lookalike': test_images_dir_look,
          'images_dir_clean': test_images_dir_clean}

input_size = 512 # Je choisis de réduire la taille des images mais nous pouvons expérimenter avec les résolutions originales (2048 pixels) aussi.

# Les points de contrôle seront stockés ici.
out_dir_model = r"C:\Users\user\Documents\oilspill\models"
# Les journaux d'exécution (tensorboard) seront stockés ici.
out_dir_runs = r"C:\Users\user\Documents\oilspill\runs"
# Si votre entraînement s'interrompt et que vous souhaitez reprendre l'entraînement.
checkpoint_to_resume = r"C:\user\Documents\oilspill\models\best_model_satlas.pt" 
# Une fois entraîné, ceci devrait représenter le modèle avec le meilleur score F1 sur les données de validation
checkpoint_best = r"C:\Users\user\Documents\oilspill\models\best_model_satlas.pt" 

# Ici je choisis de fusionner les classes clean & lookalike (classification binaire) mais nous pouvons bien sûr expérimenter avec trois classes aussi.

# Paramètres pour la classification binaire
class_names = ["clean","oil"]
num_classes = 2

# # Paramètres pour la classification à trois classes
# class_names = ["clean","oil", "lookalike"]
# num_classes = 3

## Commencer l'entraînement
Pendant l'entraînement, les journaux seront stockés via tensorboard pour le suivi de l'expérience. Utilisez `tensorboard --logdir` pour consulter l'expérience.

La progression sera également sauvegardée sous forme de graphiques.

Les points de contrôle seront sauvegardés après chaque `save_model_every` époques, et le meilleur modèle (basé sur le score F1 de validation) à chaque époque sera également sauvegardé, s'il est atteint.

In [ ]:
run_name = oilcls.main_train(train_df=train_df,
                            val_df=val_df,
                            test_df=test_df,
                            ckpt_dir=out_dir_model,
                            run_dir=out_dir_runs,
                            model_type = "satlas",
                            augment_train=True,
                            num_epochs=21,
                            batch_size=64,
                            lr=1e-3,
                            weight_decay=1e-4,
                            freeze_backbone=True,
                            freeze_backbone_partial=False,
                            use_scheduler=True,
                            save_model_every=5,
                            resume_training=False,
                            resume_weights_only=False,
                            resume_ckpt_path=checkpoint_to_resume,
                            input_size=input_size,
                            num_classes=num_classes,
                            class_labels=class_names)

## Tester la performance
Les métriques de classification du meilleur modèle entraîné seront calculées et affichées.

Les matrices de confusion seront stockées.

In [ ]:
out_dir_plots = os.path.join(out_dir_runs, "plots_" + run_name)
os.makedirs(out_dir_plots, exist_ok=True)
oilcls.main_test(train_df=train_df,
            val_df=val_df,
            test_df=test_df,
            model_path=checkpoint_best,
            plot_out_dir=out_dir_plots,
            model_type = "satlas",
            batch_size=64,
            input_size=input_size,
            num_classes=num_classes,
            class_labels=class_names)

## Inférence
Notez que bien que j'effectue l'inférence sur des exemples du jeu de données de test, les étiquettes ne sont pas utilisées.

Les sorties de l'inférence incluent :

- CSV de prédiction : Pour chaque image, la classe prédite sera indiquée avec les scores.
- CAM : Pour chaque image, une carte d'activation de classification (*Classification Activation Map [CAM]*) géoréférencée avec la même transformation/crs que l'image d'entrée sera sauvegardée.

In [ ]:
oilcls.main_infer(images_dir=test_images_dir_oil, model_path=checkpoint_best,
               out_masks_dir=os.path.join(out_dir_runs, "infer_oil"), model_type="satlas",
               batch_size=8, input_size=input_size, num_classes=num_classes, class_labels=class_names)
oilcls.main_infer(images_dir=test_images_dir_look, model_path=checkpoint_best,
           out_masks_dir=os.path.join(out_dir_runs, "infer_lookalike"), model_type="satlas",
           batch_size=8, input_size=input_size, num_classes=num_classes, class_labels=class_names)
oilcls.main_infer(images_dir=test_images_dir_clean, model_path=checkpoint_best,
           out_masks_dir=os.path.join(out_dir_runs, "infer_clean"), model_type="satlas",
           batch_size=8, input_size=input_size, num_classes=num_classes, class_labels=class_names)

### Voici des exemples de graphiques et de résultats produits par ce modèle.

Exemple de contenu du CSV de prédiction créé pendant l'inférence :

    image_path,predicted_class,score_clean,score_oil
    C:\Users\user\Documents\oilspill\dataset\test\oil\00000.tif,oil,0.33203956,0.66796046
    C:\Users\user\Documents\oilspill\dataset\test\oil\00001.tif,oil,0.2085741,0.7914259
    C:\Users\user\Documents\oilspill\dataset\test\oil\00002.tif,oil,0.30676997,0.69323003
    C:\Users\user\Documents\oilspill\dataset\test\oil\00003.tif,oil,0.0797964,0.92020357


Exemple de CAM créé pendant l'inférence : 

![](./Assets/cam_presentation_2.png)


Progression de l'entraînement capturée par Tensorboard :

![](./Assets/train_acc.png)

![](./Assets/train_loss.png)

Évaluation de la performance effectuée via la fonction main_test :

    -------------------- RÉSULTATS SUR LE JEU DE DONNÉES D'ENTRAÎNEMENT
    Précision globale : 0.8851963746223565
    Rapport de classification
                  precision    recall  f1-score   support
    
           clean       0.94      0.84      0.89       176
             oil       0.83      0.94      0.88       155
    
        accuracy                           0.89       331
       macro avg       0.89      0.89      0.89       331
    weighted avg       0.89      0.89      0.89       331
    
![](./Assets/CM_TRAIN.png)
    
    -------------------- RÉSULTATS SUR LE JEU DE DONNÉES DE VALIDATION
    Précision globale : 0.9433962264150944
    Rapport de classification
                  precision    recall  f1-score   support
    
           clean       0.96      0.93      0.95        28
             oil       0.92      0.96      0.94        25
    
        accuracy                           0.94        53
       macro avg       0.94      0.94      0.94        53
    weighted avg       0.94      0.94      0.94        53

![](./Assets/CM_VALIDATION.png)

    -------------------- RÉSULTATS SUR LE JEU DE DONNÉES DE TEST
    Précision globale : 0.8266666666666667
    Rapport de classification
                  precision    recall  f1-score   support
    
           clean       0.88      0.86      0.87       300
             oil       0.73      0.76      0.75       150
    
        accuracy                           0.83       450
       macro avg       0.80      0.81      0.81       450
    weighted avg       0.83      0.83      0.83       450

![](./Assets/CM_TEST.png)

## Exercice 1

Visualiser géographiquement les échantillons mal classés et identifier les schémas spatiaux (*patterns*) associés aux erreurs.

Comment faire cela ?

- Filtrer les échantillons mal classés
- Les tracer sur la carte
- Comparer les emplacements d'erreur avec :
    - la côte
    - les régions avec une navigation dense
    - les régions absentes des données d'entraînement
- Les diviser par leurs étiquettes d'origine `lookalike` et `clean`, et voir quel pourcentage des erreurs est causé par des confusions avec les cas lookalike ?
- Examiner le jeu de données `lookalike`, voyez-vous des possibilités d'amélioration supplémentaire ?
- Essayer la classification à trois classes et voir si les confusions diminuent.

## Exercice 2

Comparer les mêmes expériences mais avec un modèle RESNET18 pré-entraîné sur le jeu de données ImageNet.
Cette fois, dégeler (*unfreeze*) le backbone car nous savons qu'un backbone gelé ne va pas bien fonctionner.

## Aide-mémoire - Résultats Resnet

-------------------- RÉSULTATS SUR LE JEU DE DONNÉES D'ENTRAÎNEMENT

Précision globale : 0.7129909365558912

Rapport de classification

                     precision    recall  f1-score   support

           clean       0.78      0.64      0.70       176
             oil       0.66      0.79      0.72       155
    
        accuracy                           0.71       331
       macro avg       0.72      0.72      0.71       331
    weighted avg       0.72      0.71      0.71       331

Matrice de confusion :

     [[113  63]
     [ 32 123]]
     
-------------------- RÉSULTATS SUR LE JEU DE DONNÉES DE VALIDATION

Précision globale : 0.7547169811320755

Rapport de classification

                  precision    recall  f1-score   support

           clean       0.83      0.68      0.75        28
             oil       0.70      0.84      0.76        25
    
        accuracy                           0.75        53
       macro avg       0.76      0.76      0.75        53
    weighted avg       0.77      0.75      0.75        53

Matrice de confusion :

     [[19  9]
     [ 4 21]]
     
-------------------- RÉSULTATS SUR LE JEU DE DONNÉES DE TEST

Précision globale : 0.5288888888888889

Rapport de classification

                      precision    recall  f1-score   support
    
           clean       0.64      0.67      0.66       300
             oil       0.27      0.24      0.25       150
    
        accuracy                           0.53       450
       macro avg       0.45      0.46      0.45       450
    weighted avg       0.52      0.53      0.52       450

Matrice de confusion :

     [[202  98]
     [114  36]]